In [ ]:
import os
import re
import pandas as pd
import time
import datetime
from csv import reader
import csv
from dateutil import parser
import glob
from pathlib import Path
import duckdb
from dateutil.parser import parse
import warnings
from fuzzywuzzy import fuzz
import pandas_dedupe
import sys 
import json
from datetime import datetime
warnings.filterwarnings("ignore")

class Extraction:

    def __init__(self, folder_path , failed_folder_path , duplicateId_folder_path) -> None:
        self.folder_path = folder_path
        self.failed_folder_path = failed_folder_path
        self.duplicateId_folder_path = duplicateId_folder_path

    
    def saveJson(self, dictionary,filename) -> None:
        with open('./Json Files/' + filename + '.json', 'w') as fp:
            json.dump(dictionary,fp)


    def readJson(self,latest_site_id):
        with open('./Json Files/' + latest_site_id + '.json', 'r') as fp:
            data = json.load(fp)
        latest_site_id = list(data.keys())[0]
        return data , latest_site_id
    
    def getDbSizeFolder(self,filename):
        size = round(os.path.getsize(filename)/(1000*1024),2)
        return size


    def getFailityDbList(self,root_path):
        provinces = []
        districts = []
        facilities = []
        facilities_size= []
        file_sizes = []
        for folder in os.walk(root_path):
            if folder[0] =="./":
                continue 
            if len(folder[2]) == 0:
                continue

            sql_files =  [x for x in folder[2] if x.endswith('.sql')]
            if len(sql_files) ==0:
                continue

            if "\\" not in folder[0]:
                file = folder[0].replace("./","")
                provinces.append(file)
                districts.append(file)
                facilities_size.append(len(folder[2]))
                for facility in folder[2]:
                    if str(facility).endswith(".sql"):
                        file_path = "./" + file + "/"  + facility
                        facilities.append(file_path)
                        file_sizes.append(extraction.getDbSizeFolder(file_path))
                    else:
                        continue
            else:
                file = folder[0].replace("./","")
                province,district = file.split("\\")
                provinces.append(province)
                districts.append(district)
                facilities_size.append(len(folder[2]))
                for facility in folder[2]:
                    if facility.endswith(".sql"):
                        file_path = "./" + province + "/" + district + "/" + facility
                        facilities.append(file_path)
                        file_sizes.append(extraction.getDbSizeFolder(file_path))
                    else:
                        continue
            df = pd.DataFrame(list(zip(provinces, districts,facilities_size)),
                              columns =['Province', 'District','Number of Databases'])
            total_size = round(sum(file_sizes),2)
        return facilities, df, total_size


    def getFailedDbList(self):
        failed_db_list = [f for f in os.listdir(self.failed_folder_path) if f.endswith('.sql')]
        return failed_db_list
    

    def getFilesDoneList(self) -> list:
        csv_files = glob.glob('*.{}'.format('csv'))
        csv_files = [csv for csv in csv_files if csv.startswith("CBS")]
        csv_files = [sub.replace('.csv', '') for sub in csv_files ]
        return csv_files
    
    def split_string(self,string):
        # Match all commas outside of quotes
        pattern = r",(?=(?:[^']*'[^']*')*[^']*$)"
        return re.split(pattern, string)
    

    def readTables(self,folder_path,filename):
        with open(os.path.join(folder_path, filename), "r", encoding='ISO-8859-1') as f:
            content = f.read()
        content = content.replace("Unprotected sex (sex without a condom)", "Unprotected sex without a condom")
        subText = "',',"
        content = content.replace(subText,"','")
        latest_site_id = []
        latest_timestamp = []
        first_timestamp = []
        version = []
        
        domain_event = ""
        if "/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;" in content:
            domain_event = content.split("/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;")
        else:
            latest_site_id = "corrupt"
            fact_tables = {}
            row = ""
            first_timestamp =  ""
            version = ""
            return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables

        if "/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */" in domain_event[1]:
            domain_event = domain_event[1].split("/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */")
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[0])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[0])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[0])
        else:
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[1])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[1])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[1])
        if facility_search:
            latest_site_id = facility_search[-1]
            latest_timestamp = time_serach[-1]
            first_timestamp = time_serach[0]
            if len(version)==0:
                version = ""
            else:
                version = version[-1]
        db_tables = {}
        fact_tables = {}
        database_statements = re.split(r'(?<=\n)CREATE DATABASE', content)
        for statement in database_statements[1:]:
            table_names = []
            database_name = re.findall(r'`(\w+)`', statement)[0]

            if database_name not in ("client","consultation","report"):
                 continue
            # Use regular expressions to extract the table names
            pattern = re.compile(r"CREATE TABLE `(.+?)`", re.DOTALL)
            matches = pattern.findall(statement)
            
            # Loop through the matches and append the table names to the list
            for match in matches:
                table_names.append(match)

            
            # Create a dictionary of dataframes for the tables
            tables_dict = {}
            for table_name in table_names:

                # if table_name not in table_list:
                #     continue
                table_str = re.search(rf'CREATE TABLE `{table_name}`.+?ENGINE=InnoDB DEFAULT CHARSET', statement, re.DOTALL)
                if table_str is None:
                    continue
                table_str  = table_str.group(0).strip()
                columns = re.findall(r'`(\w+)` (\w+)', table_str)
                column_names = [column[0] for column in columns if not column[0].startswith("fk")]
                data = re.findall(rf'INSERT INTO `{table_name}` VALUES \(.+?\);', statement, re.DOTALL)
                rows = {}
                if data:
                    values_list = []
                    for row in data: 
                        # row = re.findall(r'\((.*?)\)(?!.*\))', row)
                        
                        row = re.findall(r'\((.*?)\)', row)
                        # row = row.split('(',1)[1]
                        
                        values2 = [ re.split(r',(?=")', s) for s in row]
            
                        for item in values2:
                            item = item[0].replace('NULL', "'NULL'")
                            rec = [i.replace("'","").lstrip() for i in item.split("',")]
                            values_list.append(rec)
                     
                    for column in column_names:
                        coulum_data = []
                        column_index = column_names.index(column)
                        for record in values_list:
                            if column_index < len(record):
                                coulum_data.append(record[column_index])
                            else:
                                coulum_data.append("")  
                        rows[column] = coulum_data
                    table_df = pd.DataFrame(rows)
                    table_df = table_df.to_dict(orient="records")
                    tables_dict[table_name] = table_df
                # else:
                #     table_df = pd.DataFrame(columns=column_names)
                #     tables_dict[table_name] = table_df
                # Add the database and its tables to the dictionary
            db_tables[database_name] = tables_dict

            if len(latest_site_id)==0:
                latest_site_id = ""
                fact_tables = {}
                row = ""
                first_timestamp =  ""
                version = ""
                return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables
            
        fact_tables[latest_site_id] = db_tables

        return latest_site_id, latest_timestamp ,first_timestamp, version,fact_tables


    def getDbContent(self, filename):
        with open(os.path.join(self.folder_path, filename), "r", encoding='ISO-8859-1') as f:
            content = f.read()
        return content
    

    def getDbSize(self,filename):
        size = round(Path(self.folder_path + filename).stat().st_size /(1000*1024),2)
        return size


    def getSiteDetails(self,content):
        latest_site_id = []
        latest_timestamp = []
        first_timestamp = []
        version = []
        domain_event = ""
        if "/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;" in content:
            domain_event = content.split("/*!40000 ALTER TABLE `domain_event_entry` DISABLE KEYS */;")
        if "/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */" in domain_event[1]:
            domain_event = domain_event[1].split("/*!40000 ALTER TABLE `domain_event_entry` ENABLE KEYS */")
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[0])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[0])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[0])
        else:
            facility_search = re.findall('ZW[a-zA-Z0-9]{6,7}',domain_event[1])
            time_serach = re.findall('[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}', domain_event[1])
            version = re.findall('[0-9]{1}\\.[0-9]{1}\\.[0-9]{2}-Q{1}[0-9]{1}',domain_event[1])
        if facility_search:
            latest_site_id = facility_search[-1]
            latest_timestamp = time_serach[-1]
            first_timestamp = time_serach[0]
            version = version[-1]
        return latest_site_id, latest_timestamp ,first_timestamp, version


    def log(self,processing_time,province="",district="",facility="",site_code="",filename="",first_date="",last_date="",
            db_size="",version="",comment="",ehr_type = ""): 
         df = pd.DataFrame({"Time File Processed":[processing_time],
                            "province": [province],
                            "district": [district],
                            "facility": [facility],
                            "Site Code":[site_code],
                            "File Name":[filename],
                            "Date site first used EHR": [first_date],
                            "Date site last used EHR": [last_date],
                            "Database Size": [db_size],
                            "Version":[version],
                            "Comments":[comment],
                            "EHR Type":[ehr_type]
                            })
         df.to_csv('log.csv', mode='a', index = False, header=None)
         
    
    def moveFailedDB(self, filename) -> None:
         Path(self.folder_path + filename).rename( self.failed_folder_path + filename)
    

    def moveDuplicatedIdDB(self, filename) -> None:
         Path(self.folder_path + filename).rename( self.duplicateId_folder_path + filename)


    def dbTableList(self):
        # List of tables to use for the program / this reduce time for table reconstruction by focusing only on tables to be used
        table_list = ["hts","hts_screening","person_investigation","patient","cbs",
                      "sexual_history","sexual_history_question","art","art_visit",
                      "art_current_status","tb","art_who_stage","person",
                      "art_appointment","laboratory_investigation_test","laboratory_request_order",
                      "art_transfer_out","patient_tb_screening","patient_client_profile"]
        return table_list

    def getMappingFile(self):
        mapping_file = pd.read_csv("mapping_file.csv")
        mapping_file['Facility ID'] = mapping_file['Facility ID'].str.strip()
        return mapping_file
    
    def getFacilityDetails(self,mapping_file,latest_site_id):
        facility_name = mapping_file.loc[mapping_file['Facility ID'] == latest_site_id] ["Facility"].values
        if facility_name.size > 0:
            facility_name = facility_name[0]
            district_name = mapping_file.loc[mapping_file['Facility ID'] == latest_site_id] ["District"].values
            if district_name.size > 0:
                district_name = district_name[0]
            else:
                district_name = ""
            province_name = mapping_file.loc[mapping_file['Facility ID'] == latest_site_id] ["Province"].values
            if province_name.size > 0:
                province_name = province_name[0]
            else:
                province_name = ""
        else:
            facility_name = ""
            district_name = ""
            province_name = ""
        return facility_name,district_name,province_name
    
    def deduplication(self,demographics):
        clusters = pandas_dedupe.dedupe_dataframe(demographics,['FirstName','LastName', 'Birthdate', 'Sex', "Street", "City","Education Id"],n_cores = 0)
        return clusters
    

    def cbs_unique_id(record,seq_number):
        cluster_id_dictionary = {}
        seq_number = 0
        if record["person_id"] == "":
            cbs_id = "No Demographics"
        elif record["cluster id"] in cluster_id_dictionary.keys():
            cbs_id = cluster_id_dictionary.get(record["cluster id"])
        else:
            first_letter_first_name = str(record["firstname"])[0]
            last_letter_first_name = str(record["firstname"])[-1]
            first_letter_last_name = str(record["Lastname"])[0]
            last_letter_last_name = str(record["Lastname"])[-1]
            first_letter_of_sex = str(record["sex"])[0]
            birthdate = str(record["birthdate"]).replace("/","")
            sequential_number = str(seq_number).rjust(6,"0")
            cbs_id = first_letter_first_name + last_letter_first_name + "-" + first_letter_last_name + last_letter_last_name + "-" + first_letter_of_sex + "-" + birthdate + "-" + sequential_number
            cluster_id_dictionary[ record["cluster id"]] = cbs_id
        return cbs_id
    

if __name__ == '__main__':

    newpath = './Json Files' 
    if not os.path.exists(newpath):
        os.makedirs(newpath)
   
    processing_time = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
    extraction  = Extraction("./", "./Failed/","./DuplicateId/")
    facilities,df_stats, total_db_size = extraction.getFailityDbList('./')
    table_list = extraction.dbTableList()
    mapping_file = extraction.getMappingFile()
    # failed_db_list = extraction.getFailedDbList()
    files_done = extraction.getFilesDoneList()
    ehr_type = "Web"

    
    user_choice = input("""Do you want to continue with previous data ? \n
                        If you choose not to continue , all your previous data will be deleted.\n
                        Click any key which is not n for Yes or click n for No""")


    if user_choice =="n":
        if len(facilities) == 0 :
            if len(files_done)==0:
                print(">>> No files for deduplication")
                sys.exit()
        else:
            if Path(extraction.folder_path+ "log.csv").exists():
                os.remove("log.csv")
            if Path(extraction.folder_path+ "demos.csv").exists():
                os.remove("demos.csv")
            for file in files_done:
                path_to_file = os.path.join(extraction.folder_path,file + ".csv")
                os.remove(path_to_file)
    
    if len(facilities)==0:
        if len(files_done)==0:
                print(">>> No files for deduplication")
                sys.exit()
    elif len(facilities) == 1:
        print(">>> Processing ", len(facilities) , " database")
        print(">>> Total Databases size ", round(total_db_size/1024))
    else:
        print(">>> Processing ", len(facilities) , " databases")
        print(">>> Total Databases size ", round(total_db_size/1024))

########## Loop through all the Facility db's ######################################
    count = 0

    reduc = 1
    for facility in facilities:
        try:
            count  = count + 1
            db_start_time = time.time()
            db_size = extraction.getDbSize(facility)
            folder_path = extraction.folder_path

            size_left =  total_db_size - db_size
            if size_left > 1000:
                print("*** ", round((size_left/1024),2), " GB left")
            else:
                print("*** ", size_left , " MB left")
            

            num_facilities = len(facilities) - reduc
            print("*** ", num_facilities , " databases left")

            total_db_size = total_db_size - db_size
            reduc = reduc + 1

            print(count ,  "/" , len(facilities))
            print("*** Working on ", facility , " start time:",datetime.now().time()  , "\n" ,
            ">>> File Size :" ,db_size, " MB")

            

            if not Path(folder_path+ "log.csv").exists():
                log_file_header = [
                    "Time File Processed",
                    "Province",
                    "District",
                    "Facility",
                    "Site Code",
                    "File Name",
                    "Date site first used EHR",
                    "Date site last used EHR",
                    "Database size /MB",
                    "E.H.R Version",
                    "Comments",
                    "EHR Type"
                ]
                with open(folder_path + "log.csv",'w') as file:
                    writer = csv.writer(file)
                    writer.writerow(log_file_header)
                
            files_already_logged2 = pd.read_csv("log.csv")
            files_already_logged = list(files_already_logged2["File Name"])
            site_already_logged = list(files_already_logged2["Site Code"])
            
            if not Path(folder_path+ "demos.csv").exists():
                demos_file_header = [
                        "person_id",
                        "firstname",
                        "Lastname",
                        "birthdate",
                        "sex",
                        "Self Identified Gender",
                        "Street",
                        "City",
                        "Education Id",
                        "Occupation Id",
                        "Religion id"
                    ]
                with open(folder_path + "demos.csv",'w') as file:
                    writer = csv.writer(file)
                    writer.writerow(demos_file_header)
        
            if facility in files_already_logged:
                print("<<<",facility ,"File already logged")
                extraction.log(
                            filename= facility,
                            processing_time= processing_time,
                            db_size=db_size,
                            comment="File already logged",
                            ehr_type= ehr_type
                            )
                # os.remove(facility)
                print("\n")
                print("------------------------------------------------------------------------------------------------------------")
                continue

            # if facility in failed_db_list:
            #     print("<<<",facility ,"File already moved to failed folder")
            #     extraction.log(
            #                 filename= facility,
            #                 processing_time= processing_time,
            #                 db_size=db_size,
            #                 comment="File already moved to failed folder",
            #                 ehr_type=ehr_type
            #                 )
            #     # os.remove(facility)
            #     print("\n")
            #     print("------------------------------------------------------------------------------------------------------------")
            #     continue

            
            if db_size < 1:
                print(">>> The database is corrupt.")
                extraction.log(processing_time= processing_time,
                            db_size=db_size,
                            filename= facility,
                            comment="The database is corrupt.",
                            ehr_type= ehr_type
                            )
                # extraction.moveFailedDB(facility)
                print("-----------------------------------------------------------------------------------------------------")
                print("\n")
                continue
            
            start_time = time.time()
            print(">>> Reading tables ....")
            content = extraction.getDbContent(facility)
            latest_site_id,last_timestamp,first_timestamp,version,fact_tables = extraction.readTables(folder_path,facility)

            if latest_site_id =="corrupt":
                print(">>> The database is corrupt.")
                extraction.log(processing_time= processing_time,
                            db_size=db_size,
                            filename= facility,
                            comment="The database is corrupt.",
                            ehr_type= ehr_type
                            )
                # extraction.moveFailedDB(facility)
                print("-----------------------------------------------------------------------------------------------------")
                print("\n")
                continue

            if not latest_site_id:
                print(">>> Database is empty")
                extraction.log(processing_time= processing_time,
                            db_size=db_size,
                            filename= facility,
                            comment="Database is empty",
                            ehr_type= ehr_type
                            )
                # extraction.moveFailedDB(facility)
                print("-----------------------------------------------------------------------------------------------------")
                print("\n")
                continue

            if latest_site_id =="":
                print(">>> Database is empty")
                extraction.log(processing_time= processing_time,
                            db_size=db_size,
                            filename= facility,
                            comment="Database is empty",
                            ehr_type= ehr_type
                            )
                # extraction.moveFailedDB(facility)
                print("-----------------------------------------------------------------------------------------------------")
                print("\n")
                continue

            print(">>> Site ID " ,latest_site_id)


            # save jason file........................................................................................................
            print(">>> saving Jason file for " + latest_site_id)
            extraction.saveJson(fact_tables, latest_site_id)
            print(">>> Done saving Json File")

            print(">>> reading saved Json File")
            fact_tables = extraction.readJson(latest_site_id)[0]
            

            facility_name , district_name,province_name =  extraction.getFacilityDetails(mapping_file,latest_site_id)

            if "CBS" + latest_site_id in extraction.getFilesDoneList() :
                
                print(">>> " ,facility ,"Database already done")
                extraction.log(processing_time,
                            province_name,
                            district_name,
                            facility_name,
                            latest_site_id,
                            facility,
                            first_timestamp,
                            last_timestamp,
                            db_size,
                            version,
                            "Database already done",
                            ehr_type= ehr_type
                            )
                # extraction.moveDuplicatedIdDB(facility)
                print("------------------------------------------------------------------------------------------------")
                print("\n")
                continue

            if latest_site_id in site_already_logged:
                print(">>> " ,facility ,"Database already logged")
                extraction.log(processing_time,
                            province_name,
                            district_name,
                            facility_name,
                            latest_site_id,
                            facility,
                            first_timestamp,
                            last_timestamp,
                            db_size,
                            version,
                            "Database already done",
                            ehr_type= ehr_type
                            )
                # extraction.moveDuplicatedIdDB(facility)
                print("------------------------------------------------------------------------------------------------")
                print("\n")
                continue

            print(">>> Reading of tables took ", round(time.time()-start_time,2),"s")

            
            # break
            start_time = time.time()
            print(">>> Constructing tables ....")

        
            if "hts" in fact_tables[latest_site_id]["consultation"]:
                hts = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["hts"])
            else:
                hts = pd.DataFrame({"patient_id":[],"laboratory_investigation_id":[],
                                    "hts_number":[],"reason_for_not_initiating_art":[],"entry_point":[],"purpose":[],
                                    "reason_for_not_performing_test":[],"entry_point_id":[],"hts_type":[],
                                    "client_already_positive":[],"client_already_on_art":[],"pregnant":[],"time":[]
                                })

            # start changes here ....................................................................

            if "patient" in fact_tables[latest_site_id]["consultation"]:
                patient = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient"])
            else:
                patient = pd.DataFrame({"patient_id":[],"time":[],"person_id":[]})

            if "hts_screening" in fact_tables[latest_site_id]["consultation"]:
                hts_screening = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["hts_screening"])
            else:
                hts_screening = pd.DataFrame({"patient_id":[],"result":[]})


            if "person_investigation" in fact_tables[latest_site_id]["consultation"]:
                person_investigation = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["person_investigation"])
            else:
                person_investigation = pd.DataFrame({"person_investigation_id":[],"date":[],"person_id":[],"result":[],"test":[]})


            if "cbs" in fact_tables[latest_site_id]["consultation"]:
                cbs = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["cbs"])
            else:
                cbs = pd.DataFrame({"date":[],"person_id":[],"been_on_prep":[]})


            if "sexual_history" in fact_tables[latest_site_id]["consultation"]:
                sexual_history = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["sexual_history"])
            else:
                sexual_history = pd.DataFrame({"date":[],"person_id":[],"sexual_history_id":[],"sexually_active":[]})


            if "sexual_history_question" in fact_tables[latest_site_id]["consultation"]:
                sexual_history_question = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["sexual_history_question"])
            else:
                sexual_history_question = pd.DataFrame({"sexual_history_id":[],"sexual_history_id":[],"question":[],"response_type":[]})


            if "patient_tb_screening" in fact_tables[latest_site_id]["consultation"]:
                tb_screening = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient_tb_screening"]) 
            else:
                tb_screening = pd.DataFrame({"presumptive":[],"patient_id":[]})


            if "patient_client_profile" in fact_tables[latest_site_id]["consultation"]:
                patient_client_profile = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["patient_client_profile"]) 
            else:
                patient_client_profile = pd.DataFrame({"client_profile":[],"person_id":[]})


            if "art_transfer_out" in fact_tables[latest_site_id]["consultation"]:
                art_transfer_out = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_transfer_out"])
            else:
                art_transfer_out = pd.DataFrame({"art_id":[],"transfer_out_date":[],"transfer_reason":[],"transfer_facility_id":[]})


            if "tb" in fact_tables[latest_site_id]["consultation"]:
                tb = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["tb"])
            else:
                tb = pd.DataFrame({"date":[],"person_id":[],"type_of_tb":[],"tb_disease_site":[],"tb_disease_type":[],"outcome":[]})


            if "laboratory_investigation_test" in fact_tables[latest_site_id]["consultation"]:
                laboratory_investigation_test = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["laboratory_investigation_test"])
            else:
                laboratory_investigation_test = pd.DataFrame({"laboratory_investigation_id":[],"laboratory_investigation_id":[],"result":[],"time":[]})


            if "laboratory_request_order" in fact_tables[latest_site_id]["consultation"]:
                lab_request = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["laboratory_request_order"])
            else:
                lab_request = pd.DataFrame({"laboratory_request_order_id":[],"laboratory_id":[]})


            if "art_visit" in fact_tables[latest_site_id]["consultation"]:
                art_visit = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_visit"])
            else:
                art_visit = pd.DataFrame({"visit_type":[],"tb_status":[],"family_planning_status":[],"lactating_status":[],"patient_id":[]})


            if "art_who_stage" in fact_tables[latest_site_id]["consultation"]:
                art_who_stage = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_who_stage"])
            else:
                art_who_stage = pd.DataFrame({"date":[],"person_id":[],"art_id":[],"stage":[],"follow_up_status":[]})

            
            if "art_current_status" in fact_tables[latest_site_id]["consultation"]:
                art_current_status =  pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_current_status"])
            else:
                art_current_status = pd.DataFrame({"date":[],"person_id":[],"art_id":[],"regimen":[],
                                                "state":[],"art_initiation_category":[],"regimen_id":[]})


            if "art_appointment" in fact_tables[latest_site_id]["consultation"]:
                art_appointment = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art_appointment"])
            else:
                art_appointment = pd.DataFrame({"date":[],"art_id":[]})


            if "art" in fact_tables[latest_site_id]["consultation"]:
                art = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["art"])
            else:
                art = pd.DataFrame({"date":[],"art_id":[],"person_id":[],"art_number":[],"art_cohort_number":[],"date_enrolled":[]})

            if "person_diagnosis" in fact_tables[latest_site_id]["consultation"]:
                person_diagnosis = pd.DataFrame(fact_tables[latest_site_id]["consultation"]["person_diagnosis"])
            else:
                person_diagnosis = pd.DataFrame({"diagnosis":[],"final_diagnosis":[],"date_created":[]})
            
            
            if "client" in fact_tables[latest_site_id]:
                if "person" in fact_tables[latest_site_id]["client"]:
                    demographics = pd.DataFrame(fact_tables[latest_site_id]["client"]["person"])
                else:
                    print(">>> " ,facility ,"Database is empty")
                    extraction.log(processing_time,
                                province_name,
                                district_name,
                                facility_name,
                                latest_site_id,
                                facility,
                                first_timestamp,
                                last_timestamp,
                                db_size,
                                version,
                                "Database is empty",
                                ehr_type= ehr_type
                                )
                    # extraction.moveFailedDB(facility)
                    print("-----------------------------------------------------------------------------------------------------")
                    print("\n")
                    continue
            else:
                print(">>> " ,facility ,"The database is corrupt.")
                extraction.log(processing_time,
                            province_name,
                            district_name,
                            facility_name,
                            latest_site_id,
                            facility,
                            first_timestamp,
                            last_timestamp,
                            db_size,
                            version,
                            "The database is corrupt.",
                            ehr_type= ehr_type
                            )
                # extraction.moveFailedDB(facility)
                print("-----------------------------------------------------------------------------------------------------")
                print("\n")
                continue


            # if "report" not in fact_tables[latest_site_id]:
            #     print(">>> " ,facility ,"The database is corrupt.")
            #     extraction.log(processing_time,
            #                 province_name,
            #                 district_name,
            #                 facility_name,
            #                 latest_site_id,
            #                 facility,
            #                 first_timestamp,
            #                 last_timestamp,
            #                 db_size,
            #                 version,
            #                 "The database is corrupt. (report database missing)",
            #                 ehr_type= ehr_type
            #                 )
            #     # extraction.moveFailedDB(facility)
            #     print("-----------------------------------------------------------------------------------------------------")
            #     print("\n")
            #     continue


            demographics["birthdate"] = pd.to_datetime(demographics["birthdate"],infer_datetime_format=True,errors='coerce')
            person_diagnosis["date_created"] = pd.to_datetime(person_diagnosis["date_created"],infer_datetime_format=True,errors='coerce')
            patient["time"] = pd.to_datetime(patient["time"],infer_datetime_format=True,errors='coerce')
            hts["time"] = pd.to_datetime(hts["time"],infer_datetime_format=True,errors='coerce')


            c_people_registered_in_ehr_sex = duckdb.query("""
                        select sex,count(*) as 'Number of people in EHR by gender' from demographics
                        group by sex
            """).df()

            c_people_registered_in_ehr_age = duckdb.query("""
                        select year(birthdate) as birthdate,count(*) as 'Number of people in EHR by age' from demographics
                        group by year(birthdate)                
            """).df()

            c_people_registered_in_ehr_nationality = duckdb.query("""
                        select nationality,count(*) as 'Number of people in EHR by nationality' from demographics
                        group by nationality
            """).df()

            c_people_registered_in_ehr_occupation = duckdb.query("""
                        select occupation,count(*) as 'Number of people in EHR by occupation' from demographics
                        group by occupation
            """).df()

            c_people_registered_in_ehr_marital = duckdb.query("""
                        select marital,count(*) as 'Number of people in EHR by Marital Status' from demographics
                        group by marital
            """).df()

            c_people_registered_in_ehr_religion= duckdb.query("""
                        select religion,count(*) as 'Number of people in EHR by religion' from demographics
                        group by religion
            """).df()

            c_people_registered_in_ehr_education = duckdb.query("""
                        select education,count(*) as 'Number of people in EHR by education' from demographics
                        group by education
            """).df()

            replace_dict = {"\x01":1,"\\0":0,"":0}
            person_diagnosis.replace(replace_dict, inplace=True)

            c_diagnosis = duckdb.query("""
                        SELECT diagnosis,count(*) as 'Total Number of people diagnosed' FROM person_diagnosis
                        where final_diagnosis = 1
                        group by diagnosis
            """).df()



            c_diagnosis_by_year = duckdb.query("""
                        SELECT year(date_created) as 'Year', diagnosis, count(*)  as 'Number of people diagnosed' FROM person_diagnosis
                        where final_diagnosis = 1
                        group by diagnosis, year(date_created)
                        order by year(date_created) , count(*) desc
            """).df()

            c_distinct_patients_attended_by_year = duckdb.query("""
                        SELECT year(time) as 'Year',count(distinct person_id) as 'Number of distinct patients attended' FROM patient
                        group by year(time)         
            """).df()

            c_patients_attended_by_year = duckdb.query("""
                        SELECT year(time) as 'Year',count(person_id) as 'Number of patients attended' FROM patient
                        group by year(time)         
            """).df()

            c_distinct_people_who_had_hts = duckdb.query("""
                        select year(hts.time) as 'Year',count(distinct patient.person_id) as 'Number of people in HTS'
                        from hts left join patient
                        on hts.patient_id = patient.patient_id
                        group by year(hts.time)
                        ;         
            """).df()

            # HTS ....................................................................................................................
            df_hts = duckdb.query("""
                            WITH ranked_messages AS (
                            select 
                            pt.time as Event_date,
                            p.person_investigation_id, p.person_id,h.patient_id, h.laboratory_investigation_id,
                            h.hts_number as 'HTS Number',reason_for_not_initiating_art,
                            pt.time as 'Date of HIV Test',h.entry_point as 'Entry Point',h.purpose ,
                            h.reason_for_not_performing_test as 'Reason for not doing recency test',
                            h.entry_point_id as 'Entry Point Id', h.hts_type as 'Test Method', 
                            h.client_already_positive as 'Already positive',
                            h.client_already_on_art as 'Already on art', p.result as HIV_Result,
                            h.pregnant as 'Pregnant during HIV Test',
                            ROW_NUMBER() OVER (PARTITION BY p.person_id ORDER BY pt.time ASC) as rnt
                            FROM hts h  
                            left join patient pt
                            on  h.patient_id = pt.patient_id
                            left join person_investigation p
                            on h.laboratory_investigation_id = p.person_investigation_id
                            where  p.test = 'HIV' and p.result = 'POSITIVE' and h.purpose <> 'Retesting for ART initiation'
                            )
                            SELECT * FROM ranked_messages WHERE rnt = 1
                        """
                        ).df()

            df_hts.rename(columns = {'purpose':'Reason for HIV Test'}, inplace = True)

            
            
            if df_hts.empty:
                print(">>> There are no positive cases in the database")
                extraction.log(processing_time,
                            province_name,
                            district_name,
                            facility_name,
                            latest_site_id,
                            facility,
                            first_timestamp,
                            last_timestamp,
                            db_size,
                            version,
                            "There are no positive cases in the database",
                            ehr_type= ehr_type
                            )
                # os.remove(facility)
                print("-----------------------------------------------------------------------------------------------------")
                print("\n")
                continue
            
            
            df_hts_investigations =  duckdb.query(
                        """
                            WITH ranked_messages AS(
                            select  h.laboratory_investigation_id, h.result,h.time , p.person_investigation_id, p.person_id,
                            ROW_NUMBER() OVER (PARTITION BY p.person_investigation_id ORDER BY time ASC) as rtt
                            from person_investigation p
                            left join laboratory_investigation_test h
                            on p.person_investigation_id = h.laboratory_investigation_id
                            )
                            select * from ranked_messages 
                            where person_investigation_id in 
                            ( select person_investigation_id from df_hts)
                        """
                        ).df()
            
            
            df_hts_investigations['rtt'] = df_hts_investigations['rtt'].map({1:"HIVtestOneResult", 2: "HIVtestTwoResult",3:"HIVtestThreeResult"})
            df_hts_investigations.drop_duplicates(subset=["person_id","rtt"],keep= 'first', inplace=True)
            df_hts_investigations = df_hts_investigations.pivot(index='person_id', columns=['rtt'], values='result')
            df_hts_investigations = df_hts_investigations.reset_index()


            # Extract date of last known hiv negative test from hts.....................................................
            df_hts_negative = duckdb.query("""
                            WITH ranked_messages AS (
                            select 
                            pt.time as Event_date,
                            p.person_id,h.patient_id,
                            pt.time as 'Last HIV negative date',
                            ROW_NUMBER() OVER (PARTITION BY p.person_id ORDER BY pt.time desc) as rnt
                            FROM hts h  
                            left join patient pt
                            on  h.patient_id = pt.patient_id
                            left join person_investigation p
                            on h.laboratory_investigation_id = p.person_investigation_id
                            where  p.test = 'HIV' and p.result = 'NEGATIVE' 
                            )
                            SELECT * FROM ranked_messages WHERE rnt = 1
                        """
                        ).df()


            df_hts_merge_first = pd.merge(df_hts,df_hts_investigations, on =["person_id"], how ="left")
            df_hts_merge = pd.merge(df_hts_merge_first,df_hts_negative, on =["person_id","Event_date"], how ="left")

            df_hts_merge["HTS Province Name"] = province_name
            df_hts_merge["HTS District Name"] = district_name
            df_hts_merge["HTS Facility Name"] = facility_name
            df_hts_merge["HTS Facility Id"] = latest_site_id
        
        
            column_order = {'Event_date': 1, 'person_id': 2, 
                            'HTS Number': 3 , 'HTS Province Name':4,
                            'HTS District Name':5,'HTS Facility Name':6,'HTS Facility Id':7,
                    'Date of HIV Test':8,'Reason for HIV Test':9,'Entry Point':10,'Entry Point Id':11, 
                    'Pregnant during HIV Test':12, 
                    'Already positive':13, 'Already on art':14, 'Test Method':15, 
                    'Last HIV negative date':16,
                    'HIVtestOneResult':17,
                    'HIVtestTwoResult':18,'HIVtestThreeResult':19, 'HIV_Result':20,'Reason for not doing recency test':21,
                    'reason_for_not_initiating_art':22
                            }
            df_hts_merge = df_hts_merge[[col for col in sorted(df_hts_merge.columns, key=lambda x: column_order.get(x, float('inf')))]]

            df_hts_merge.rename(columns = {"HIV_Result":"Final HIV Result" ,
                                        "reason_for_not_initiating_art":"Reason for not initiating on ART"  },inplace = True)
            replace_dict = {"\x01":"YES","\\0":"NO"}
            df_hts_merge.replace(replace_dict, inplace=True)

            df_hts_merge["Date of HIV Test"] = pd.to_datetime(df_hts_merge["Date of HIV Test"],errors='coerce').dt.date
            df_hts_merge["Event_date"] = pd.to_datetime(df_hts_merge["Event_date"],errors='coerce').dt.date
            df_hts_merge["Last HIV negative date"] = pd.to_datetime(df_hts_merge["Last HIV negative date"],errors='coerce').dt.date

            df_hts_merge = df_hts_merge.sort_values(by=['Final HIV Result'])
            df_hts_merge.drop_duplicates(subset=['Event_date', 'HTS Number','person_id','Date of HIV Test'], keep='last',inplace=True)
            
            # sexual history.............................................................................................................
            df_sexual_history = duckdb.query(
            """
                select sexual_history_id,person_id,sexually_active,date from sexual_history
            """
            ).df()
            sexual_history.replace({"sexually_active": replace_dict},inplace=True)
            df_sexual_history= sexual_history[['sexual_history_id', 'person_id',  'sexually_active','date']]
            

            # sexual_history_question.....................................................................................................
            df_sexual_history_question = duckdb.query(
                """
                    select sexual_history_id,question,response_type 
                    from sexual_history_question 
                """
            ).df()

            df_sexual_history_question.drop_duplicates(subset=["sexual_history_id","question"],inplace = True)
            df_sexual_history_question = df_sexual_history_question.pivot(index='sexual_history_id', columns='question')
            df_sexual_history_question.columns = df_sexual_history_question.columns.droplevel()
            df_sexual_history_question.reset_index(inplace=True)
            df_sexual_history_question = pd.merge(df_sexual_history_question, df_sexual_history , how="left", on=["sexual_history_id"])
            df_sexual_history_question.drop(['sexual_history_id'], axis=1,inplace=True)
            df_sexual_history_question.rename(columns = {'Exchanged sex for  money/material goods':'Exchanged sex for moneymaterial goods',
                                                        'Victim/ Suspected victim of sexual abuse':'Victim Suspected victim of sexual abuse',
                                                        'Unprotected sex without a condom':'Unprotected sex without a condom',
                                                        'sexually_active': 'Sexually Active' }, inplace = True)

            column_order = {'Event_date': 1, 
                            'person_id':2,'Been incarcerated into jail':3,
                    'Exchanged sex for moneymaterial goods':4,
                    'Had Anal Sex':5,'Had sex with male':6,'Had sex with female':7,'Had sex with a sex worker':8,
                    'Victim Suspected victim of sexual abuse':9,'Tattooed with unsterilized instruments':10,
                    'Received medical injections':11, 'Unprotected sex without a condom':12, 'Injected recreational drugs':13,
                    'History of an STI':14, 'Had Oral Sex':15,  'Received blood transfusions':16, 'Sexually Active':17
                            }
            df_sexual_history_question = df_sexual_history_question[[col for col in sorted(df_sexual_history_question.columns, key=lambda x: column_order.get(x, float('inf')))]]


            df_sexual_history_question.drop_duplicates(subset=['person_id'], keep='last',inplace=True)
        

            # CBS.....................................................................................................................
            df_cbs = duckdb.query(
            """
            select date as 'Event_date' ,
            person_id,'Yes' as 'Notified', date as 'Date of Notification' , been_on_prep as 'Been On Prep' 
            from cbs 
            where person_id in (select person_id from df_hts_merge)
            """
            ).df()

            df_cbs["CBS Province Name"] = province_name
            df_cbs["CBS District Name"] = district_name
            df_cbs["CBS Facility Name"] = facility_name
            df_cbs["CBS Facility id"] = latest_site_id
            df_cbs = df_cbs[[
                'Event_date',
                "person_id", 'Notified',
                "CBS Province Name","CBS District Name","CBS Facility Name","CBS Facility id",'Date of Notification'
            ]]
            df_cbs["Date of Notification"] = pd.to_datetime(df_cbs["Date of Notification"],errors='coerce').dt.date
            df_cbs["Event_date"] = pd.to_datetime(df_cbs["Event_date"],errors='coerce').dt.date

            df_cbs.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)



            # Recency.................................................................................................................
            df_recency = duckdb.query(
            """
            select date as 'Event_date',
            person_id, date as 'Date Recency Done',
            result as 'Recency Result'
            from person_investigation 
            where test = 'HIV-1 Rapid Recency' and person_id in (select person_id from df_hts_merge)
            """).df()
            df_recency["Date Recency Done"] = pd.to_datetime(df_recency["Date Recency Done"],errors='coerce').dt.date
            df_recency["Event_date"] = pd.to_datetime(df_recency["Event_date"],errors='coerce').dt.date
            df_recency["Recency Province Name"] = province_name
            df_recency["Recency District Name"] = district_name
            df_recency["Recency Facility Name"] = facility_name
            df_recency["Recency Facility id"] = latest_site_id
            df_recency = df_recency [[
                'Event_date',
                "person_id","Recency Province Name","Recency District Name","Recency Facility Name","Recency Facility id",
                'Date Recency Done' ,'Recency Result'
            ]]
            df_recency = df_recency.sort_values(by=['Recency Result'])
            df_recency.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


            # Art......................................................................................................................... 
            df_art = duckdb.query(
                """
                    select  date as 'Event_date',
                    art_id,person_id,art_number as "Art Number",art_cohort_number as "Art Cohort Number",
                    date as "Art_Initiation_Date",date_enrolled as "Date Enrolled into ART"
                    from art where person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_art["ART Province Name"] = province_name
            df_art["ART District Name"] = district_name
            df_art["ART Facility Name"] = facility_name
            df_art["ART Facility id"] = latest_site_id
            df_art["Art_Initiation_Date"] = pd.to_datetime(df_art["Art_Initiation_Date"],errors='coerce').dt.date
            df_art["Event_date"] = pd.to_datetime(df_art["Event_date"],errors='coerce').dt.date


            df_art = df_art [[
                'Event_date',
                "person_id","ART Province Name","ART District Name","ART Facility Name","ART Facility id",
                "Art Number", "Art Cohort Number", "Art_Initiation_Date","Date Enrolled into ART"
            ]]
            df_art = df_art.sort_values(by=['Art Number'])
            df_art.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


            # Art Visit....................................................................................................................
            df_art_visit = duckdb.query(
                """
                    select p.time as 'Event_date', 
                    p.time as "Art visit date", p.person_id, a.visit_type as 'Art visit type',
                    a.tb_status as 'TB Status', a.family_planning_status, a.lactating_status
                    from art_visit a left join patient p
                    on a.patient_id = p.patient_id
                    where p.person_id in (select person_id from df_hts_merge)
                """
            ).df() 
            df_art_visit["ART Visit Province Name"] = province_name
            df_art_visit["ART Visit District Name"] = district_name
            df_art_visit["ART Visit Facility Name"] = facility_name
            df_art_visit["ART Visit Facility id"] = latest_site_id
            df_art_visit["Art visit date"] = pd.to_datetime(df_art_visit["Art visit date"],errors='coerce').dt.date
            df_art_visit["Event_date"] = pd.to_datetime(df_art_visit["Event_date"],errors='coerce').dt.date
            df_art_visit = df_art_visit[[
                'Event_date',
                'person_id',
                "ART Visit Province Name",
                "ART Visit District Name",
                "ART Visit Facility Name",
                "ART Visit Facility id",
                "Art visit date",'Art visit type',
                'TB Status'
            ]]

            df_art_visit = df_art_visit.sort_values(by=['Art visit type'])
            df_art_visit.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
            df_art_visit = df_art_visit.replace('Presumptive-if there are signs', 'Suspect or Presumptive')
            df_art_visit = df_art_visit.replace('Screened and has no signs', 'Screened with no signs')
            df_art_visit = df_art_visit.replace('Tb status not assesssed', '')

            


            # Art Current Status................................................................................................................
            df_art_current_status = duckdb.query(
                """
                    select a.date as 'Event_date', 
                    a.date as 'Art Current Status Date',a.art_id ,a.regimen as regimen , a.state as 'ARV Status', a.art_initiation_category as 'Art Initiation Category',
                    art.person_id,a.regimen_id as 'Regimen Id'
                    from art_current_status a 
                    left join art 
                    on a.art_id = art.art_id
                    where art.person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_art_current_status["Art Current Status Date"] = pd.to_datetime(df_art_current_status["Art Current Status Date"],errors='coerce').dt.date
            df_art_current_status["Event_date"] = pd.to_datetime(df_art_current_status["Event_date"],errors='coerce').dt.date
            df_art_current_status = df_art_current_status[[
            'Event_date','person_id', 'ARV Status','Regimen Id','regimen','Art Initiation Category'
            ]]
            df_art_current_status = df_art_current_status.sort_values(by=['regimen'])
            df_art_current_status.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)
            

            #Art who stage..............................................................................................................
            df_art_who_stage = duckdb.query(
                """
                select  a.date as 'Event_date',
                p.person_id , a.art_id, a.date as 'Who Stage Date',
                a.date as 'Outcome Date',
                a.stage as 'Who Stage', a.follow_up_status as 'Art Outcome'
                from art_who_stage a left join art p
                on a.art_id  = p.art_id
                where p.person_id in (select person_id from df_hts_merge)

                """
            ).df()
            df_art_who_stage["Who Stage Province Name"] = province_name
            df_art_who_stage["Who Stage District Name"] = district_name
            df_art_who_stage["Who Stage Facility Name"] = facility_name
            df_art_who_stage["Who Stage Facility id"] = latest_site_id
            df_art_who_stage["Who Stage Date"] = pd.to_datetime(df_art_who_stage["Who Stage Date"],errors='coerce').dt.date
            df_art_who_stage["Event_date"] = pd.to_datetime(df_art_who_stage["Event_date"],errors='coerce').dt.date
            df_art_who_stage = df_art_who_stage [[
                'Event_date',
                'person_id',"Who Stage Province Name","Who Stage District Name","Who Stage Facility Name","Who Stage Facility id",'Who Stage Date','Who Stage',
                'Art Outcome'
            ]]

            df_art_who_stage = df_art_who_stage.sort_values(by=['Who Stage'])
            df_art_who_stage.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


            # Viral Load.......................................................................................................................
            df_viral_load = duckdb.query(
                """
                    select date as 'Event_date',
                    person_id,date as "Date at which Viral Load result was issued",
                    date as "Date for which Viral Load was taken",
                    'TRUE' as "Viral Load Sample submitted to lab",
                    'TRUE' as "Was Viral Load result issued",
                    result as "Viral Load result",
                    from person_investigation
                    where test = 'Viral Load' and person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_viral_load["Viral Load Province Name"] = province_name
            df_viral_load["Viral Load District Name"] = district_name
            df_viral_load["Viral Load Facility Name"] = facility_name
            df_viral_load["Viral Load Facility id"] = latest_site_id
            df_viral_load["Date at which Viral Load result was issued"] = pd.to_datetime(df_viral_load["Date at which Viral Load result was issued"],errors='coerce').dt.date
            df_viral_load["Event_date"] = pd.to_datetime(df_viral_load["Event_date"],errors='coerce').dt.date
            df_viral_load = df_viral_load[[
                'Event_date',
                'person_id',"Viral Load Province Name","Viral Load District Name","Viral Load Facility Name","Viral Load Facility id",
                "Date at which Viral Load result was issued",
                "Date for which Viral Load was taken",
                "Viral Load Sample submitted to lab",
                "Was Viral Load result issued",
                "Viral Load result"
            ]]

            df_viral_load = df_viral_load.sort_values(by=['Viral Load result'])
            df_viral_load.drop_duplicates(subset=['Event_date','person_id'], keep='last', inplace = True)


            # cd4..............................................................................................................................
            df_cd4 = duckdb.query(
                """
                    select date as 'Event_date',
                    person_id, date as 'Date at which cd4 sample was taken',
                    date as 'Date at which cd4 result was issued',
                    'TRUE' as "CD4 Sample submitted to lab",
                    'TRUE' as "Was cd4 result issued",
                    result as 'CD4 Count'
                    from person_investigation
                    where test = 'CD4 Count' and person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_cd4["CD4 Province Name"] = province_name
            df_cd4["CD4 District Name"] = district_name
            df_cd4["CD4 Facility Name"] = facility_name
            df_cd4["CD4 Facility id"] = latest_site_id
            df_cd4["Date at which cd4 result was issued"] = pd.to_datetime(df_cd4["Date at which cd4 result was issued"],errors='coerce').dt.date
            df_cd4["Event_date"] = pd.to_datetime(df_cd4["Event_date"],errors='coerce').dt.date
            df_cd4 = df_cd4 [[
                'Event_date',
                'person_id',"CD4 Province Name","CD4 District Name","CD4 Facility Name","CD4 Facility id",
                'Date at which cd4 sample was taken',
                'Date at which cd4 result was issued',
                "CD4 Sample submitted to lab",
                "Was cd4 result issued",
                'CD4 Count',
            ]]
            df_cd4 = df_cd4.sort_values(by=['CD4 Count'])
            df_cd4.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)


            #TB..........................................................................................................................
            df_TB = duckdb.query(
                """
                    select date as 'Event_date',
                    person_id,date as 'TB Treatment Start Date',type_of_tb as 'Type Of TB',
                    tb_disease_site as 'TB Disease Site',
                    tb_disease_type as 'TB Disease Type',outcome as 'TB Outcome'
                    from tb
                    where  person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_TB["TB Province Name"] = province_name
            df_TB["TB District Name"] = district_name
            df_TB["TB Facility Name"] = facility_name
            df_TB["TB Facility id"] = latest_site_id
            df_TB["TB Treatment Start Date"] = pd.to_datetime(df_TB["TB Treatment Start Date"],errors='coerce').dt.date
            df_TB["Event_date"] = pd.to_datetime(df_TB["Event_date"],errors='coerce').dt.date

            df_TB = df_TB[[
                'Event_date',
                'person_id',"TB Province Name","TB District Name","TB Facility Name","TB Facility id",'TB Treatment Start Date',
                'TB Outcome','Type Of TB','TB Disease Site','TB Disease Type'
            ]]
            df_TB = df_TB.sort_values(by=['TB Outcome'])
            df_TB.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
            

            # TB Screening
            df_TB_Screening = duckdb.query(
                """
                    select t.time as 'Event_date',
                    t.person_id , p.presumptive as 'TB Screened', 
                    from tb_screening p
                    left join patient t 
                    on p.patient_id = t.patient_id
                    where person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_TB_Screening["TB Screening Province Name"] = province_name
            df_TB_Screening["TB Screening District Name"] = district_name
            df_TB_Screening["TB Screening Facility Name"] = facility_name
            df_TB_Screening["TB Screening Facility id"] = latest_site_id
            df_TB_Screening = df_TB_Screening.replace('\x01', 'Suspect or Presumptive')
            df_TB_Screening = df_TB_Screening.replace('\\0', 'Screened with no signs')

            df_TB_Screening = df_TB_Screening[[
                'Event_date',
                'person_id',"TB Screening Province Name","TB Screening District Name","TB Screening Facility Name","TB Screening Facility id",
                'TB Screened'
            ]]
            df_TB_Screening["Event_date"] = pd.to_datetime(df_TB_Screening["Event_date"],errors='coerce').dt.date
            df_TB_Screening.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace =  True)
            
            
            # Demographics..............................................................................................................
            df_demographics = duckdb.query(
                """
                    select 
                    person_id, birthdate,sex, self_identified_gender as 'Self Identified Gender',
                    marital_id as 'Marital Id', marital as 'Marital Status',education_id as 'Education Id',
                    education as 'Education',occupation_id as 'Occupation Id' ,occupation as 'Occupation',
                    religion_id as 'Religion id', religion as 'Religion',nationality_id as 'Nationality Id',
                    nationality as Nationality, town_id as 'Residential Town Id', town as 'Residential Town'
                    from demographics where person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_demographics = df_demographics.sort_values(by=['birthdate'])
            df_demographics.drop_duplicates(subset=['person_id'], keep='last',inplace = True)

            df_demographics_for_dedup = duckdb.query(
                    """
                    select 
                    person_id, firstname, lastname as Lastname, birthdate, sex,
                    self_identified_gender as 'Self Identified Gender', street as 'Street',
                    city as City, education_id as 'Education Id', occupation_id as 'Occupation Id',
                    religion_id as 'Religion id'
                    from demographics where person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_demographics_for_dedup = df_demographics_for_dedup.sort_values(by=['birthdate'])
            df_demographics_for_dedup.drop_duplicates(subset=['person_id'], keep='last',inplace=  True)


            df_transfer_out = duckdb.query(
                    """
                    select 
                    a.person_id , t.art_id , t.transfer_out_date as 'Event_date',
                    t.transfer_out_date as 'Transfer Out date',
                    t.transfer_reason as 'Transfer Reason', t.transfer_facility_id
                    from art_transfer_out t left join art a
                    on t.art_id = a.art_id
                    where a.person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_transfer_out["Event_date"] = pd.to_datetime(df_transfer_out["Event_date"],errors='coerce').dt.date
            df_transfer_out = df_transfer_out.sort_values(by=['transfer_facility_id'])
            df_transfer_out.drop_duplicates(subset=['Event_date','person_id'], keep='last',inplace = True)

            df_patient_client_profile = duckdb.query(
                    """
                    select 
                    person_id, client_profile as 'Client Profile'
                    from patient_client_profile
                    where person_id in (select person_id from df_hts_merge)
                """
            ).df()
            df_patient_client_profile.drop_duplicates(subset=['person_id'], keep='last',inplace = True)


            # Merging////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
            print(">>> Merging data from tables ....")  

            ### Format1 ######## Services ###############################################################################################
            #.......HTS Service .............................................................
            #hts_sexual_history = pd.merge(df_hts_merge,df_sexual_history_question, on =["person_id"] , how = "left" )

        
            # Merge HTS with risk questions (sexual history)........
            hts_sexual_history = pd.merge(df_hts_merge,df_sexual_history_question, on =["person_id"] , how = "left" )
            

            # Merge with cbs....................
            merge_cbs = pd.merge(hts_sexual_history,df_cbs, on =["Event_date","person_id"] , how = "outer" )
            

            # Merge with recency.................
            merge_recency = pd.merge(merge_cbs, df_recency, on =["Event_date","person_id"] , how = "outer" )
            
            # Merge with Art .....................
            merge_art = pd.merge(merge_recency, df_art, on =["Event_date","person_id"] , how = "outer" )
        

            # Merge with Art Visit ................
            merge_art_visit = pd.merge(merge_art, df_art_visit, on =["Event_date","person_id"] , how = "outer" )

            
            # Merge with art_current_status.........
            merge_art_current_status = pd.merge(merge_art_visit, df_art_current_status, on =["Event_date","person_id"] , how = "outer" )
            
            
            # Merge with who stage....................
            merge_who_stage = pd.merge(merge_art_current_status, df_art_who_stage, on =["Event_date","person_id"] , how = "outer" )


            # Merge with viral load....................
            merge_viral_load = pd.merge(merge_who_stage, df_viral_load, on =["Event_date","person_id"] , how = "outer" )
            
        
            # Merge with cd4...........................
            merge_cd4 = pd.merge(merge_viral_load, df_cd4 , on =["Event_date","person_id"] , how = "outer" )


            # Merge with TB Screening ...................
            merge_tb_screening = pd.merge(merge_cd4, df_TB_Screening , on =["Event_date","person_id"] , how = "outer" )
            

            # Merge with TB...............................
            merge_tb = pd.merge(merge_tb_screening, df_TB , on =["Event_date","person_id"] , how = "outer" )
            
        
            # Merge with transfer out.....................
            merge_transfer_out = pd.merge(merge_tb, df_transfer_out , on =["Event_date","person_id"] , how = "outer" )


            # Merge with demographics ....................
            merge_demographics = pd.merge(merge_transfer_out, df_demographics , on =["person_id"] , how = "outer" )

            # Merge with demographics ....................
            merge_client_profile = pd.merge(merge_demographics, df_patient_client_profile , on =["person_id"] , how = "outer" )


            merge_client_profile["Province"] = [province_name]*len(merge_client_profile)
            merge_client_profile["District"] = [district_name]*len(merge_client_profile)
            merge_client_profile["Facility"] = [facility_name]*len(merge_client_profile)
            merge_client_profile["Facility Id"] = [latest_site_id]*len(merge_client_profile)


            # Add age at event
            merge_client_profile["Age at visit"] = (pd.to_datetime(merge_client_profile["Event_date"],errors='coerce') - \
                                                    pd.to_datetime(merge_client_profile["birthdate"],errors='coerce')).dt.days //365
        
            
            merged = merge_client_profile

            merged = merged.replace(['NULL'], '')
            merged.fillna('', inplace=True)

            ## Add TB Screening column
            def tb_screen(x):
                if x["TB Status"] == '':
                    return x["TB Screened"]
                else:
                    return x["TB Status"]
            
            merged["TB Screening"] = merged.apply(tb_screen,axis = 1)
            merged.drop(['TB Status','TB Screened'], axis=1)
            
            column_order_final = {
                        'Event_date':1, 'Age at visit':2,
                        "Province":3,"District":4,"Facility":5,"Facility Id":6,
                        'person_id':7,'Client Profile':8, 'birthdate':9, 'sex':10, 'Self Identified Gender':11, 
                        'Marital Status':12,  'Education':13, 
                        'Occupation':14, 'Religion':15, 
                        'Nationality':16, 'Residential Town':17, 'HTS Number':18,
                        'HTS Province Name':19, 'HTS District Name':20, 'HTS Facility Name':21,
                        'HTS Facility Id':22, 'Date of HIV Test':23, 'Reason for HIV Test':24,
                        'Entry Point':25, 'Pregnant during HIV Test':26,
                        'Already positive':27, 'Already on art':28, 'Test Method':29, 
                        'Last HIV negative date':30,
                        'HIVtestOneResult':31,
                        'HIVtestTwoResult':32, 'HIVtestThreeResult':33,'Final HIV Result':34,
                        'Been incarcerated into jail':35,
                        'Exchanged sex for moneymaterial goods':36, 'Had Anal Sex':37,
                        'Had sex with male':38, 'Had sex with female':39, 'Had sex with a sex worker':40,
                        'Victim Suspected victim of sexual abuse':41,
                        'Tattooed with unsterilized instruments':42, 'Received medical injections':43,
                        'Unprotected sex without a condom':44, 'Injected recreational drugs':45,
                        'History of an STI':46, 'Had Oral Sex':47, 'Received blood transfusions':48,
                        'Sexually Active':49,'Notified':50,'CBS Province Name':51, 'CBS District Name':52,
                        'CBS Facility Name':53, 'CBS Facility id':54, 'Date of Notification':55,'Recency Province Name':56,
                        'Recency District Name':57, 'Recency Facility Name':58, 'Recency Facility id':59,
                        'Date Recency Done':60, 'Recency Result':61,'Reason for not doing recency test':62,
                        'ART Province Name':63, 'ART District Name':64,
                        'ART Facility Name':65, 'ART Facility id':66, 'Art Number':67,
                        'Art Cohort Number':68, 'Art_Initiation_Date':69, 'Date Enrolled into ART':70,'Reason for not initiating on ART':71,
                        'ART Visit Province Name':72,
                        'ART Visit District Name':73, 'ART Visit Facility Name':74,
                        'ART Visit Facility id':75, 'Art visit date':76, 'Art visit type':77,
                        'ARV Status':78, 'regimen':79,
                        'Art Initiation Category':80,
                        'Who Stage Province Name':81,
                        'Who Stage District Name':82, 'Who Stage Facility Name':83,
                        'Who Stage Facility id':84, 'Who Stage Date':85, 'Who Stage':86, 
                        'Viral Load Province Name':87, 'Viral Load District Name':88,
                        'Viral Load Facility Name':89, 'Viral Load Facility id':90,
                        'Date for which Viral Load was taken':91,
                        'Viral Load Sample submitted to lab':92,
                        'Viral Load result received date':93,
                        'Was Viral Load result issued':94,
                        'Date at which Viral Load result was issued':95,
                        'Viral Load result':96,
                        'CD4 Province Name':97, 'CD4 District Name':98,
                        'CD4 Facility Name':99, 'CD4 Facility id':100,
                        'Date at which cd4 sample was taken':101,
                        'Date at which cd4 result was issued':102, 'CD4 Sample submitted to lab':103,
                        'Was cd4 result issued':104, 'CD4 Count':105,
                        'TB Screening Province Name':106,'TB Screening District Name':107,
                        'TB Screening Facility Name':108,'TB Screening Facility id':109,'TB Screening':110,
                        'TB Province Name':111, 'TB District Name':112,
                        'TB Facility Name':113, 'TB Facility id':114, 'TB Treatment Start Date':115,
                        'TB Outcome':116, 'Type Of TB':117, 'TB Disease Site':118, 'TB Disease Type':119,
                        'Art Outcome':120,
                        'Transfer Out date':121,
                        'Transfer Reason':122,
                        'transfer_facility_id':123
                            }
            merged = merged[[col for col in sorted(merged.columns, key=lambda x: column_order_final.get(x, float('inf')))]]

            merged = merged.replace(['NULL'], '')
            merged = merged.replace('\x01', 'YES')
            merged = merged.replace('\\0', 'NO')
            merged.rename(columns = {'Event_date':'Event date'}, inplace = True)


            print(">>> Saving  data ....")  
            #  Aggregate files
            c_people_registered_in_ehr_sex["Facility Id"] = latest_site_id
            c_people_registered_in_ehr_education["Facility Id"]  = latest_site_id
            c_people_registered_in_ehr_marital["Facility Id"] = latest_site_id
            c_people_registered_in_ehr_nationality["Facility Id"] = latest_site_id
            c_people_registered_in_ehr_age["Facility Id"] = latest_site_id
            c_people_registered_in_ehr_occupation["Facility Id"] = latest_site_id
            c_people_registered_in_ehr_religion["Facility Id"] = latest_site_id
            
            c_diagnosis["Facility Id"] = latest_site_id
            c_diagnosis_by_year["Facility Code"] = latest_site_id
            c_patients_attended_by_year["Facility Id"] = latest_site_id
            c_distinct_patients_attended_by_year["Facility Id"] = latest_site_id
            c_distinct_people_who_had_hts["Facility Id"] = latest_site_id
            # c_distinct_people_who_had_hts_positive["Facility Id"] = latest_site_id

            c_people_registered_in_ehr_sex.to_csv("./Aggregates/c_people_registered_in_ehr_sex_" + latest_site_id + ".csv")
            c_people_registered_in_ehr_education.to_csv("./Aggregates/c_people_registered_in_ehr_education_"+ latest_site_id+".csv")
            c_people_registered_in_ehr_marital.to_csv("./Aggregates/c_people_registered_in_ehr_marital_"+latest_site_id+".csv")
            c_people_registered_in_ehr_nationality.to_csv("./Aggregates/c_people_registered_in_ehr_nationality_"+latest_site_id+".csv")
            c_people_registered_in_ehr_age.to_csv("./Aggregates/c_people_registered_in_ehr_age_"+latest_site_id+".csv")
            c_people_registered_in_ehr_occupation.to_csv("./Aggregates/c_people_registered_in_ehr_occupation_"+latest_site_id+".csv")
            c_people_registered_in_ehr_religion.to_csv("./Aggregates/c_people_registered_in_ehr_religion_"+latest_site_id+".csv")
            c_diagnosis.to_csv("./Aggregates/c_diagnosis_"+latest_site_id+".csv")
            c_diagnosis_by_year.to_csv("./Aggregates/c_diagnosis_by_year_"+ latest_site_id+".csv")
            c_patients_attended_by_year.to_csv("./Aggregates/c_patients_attended_by_year_"+latest_site_id+".csv")
            c_distinct_patients_attended_by_year.to_csv("./Aggregates/c_distinct_patients_attended_by_year_"+latest_site_id+".csv")
            c_distinct_people_who_had_hts.to_csv("./Aggregates/c_distinct_people_who_had_hts_"+latest_site_id+".csv")
            # c_distinct_people_who_had_hts_positive.to_csv("./Aggregates/c_distinct_people_who_had_hts_positive_"+latest_site_id+".csv")

            fullpath = "CBS"+ latest_site_id  + ".csv"

            merged.to_csv(fullpath,index=False)

            df_demographics_for_dedup.to_csv('demos.csv', mode='a', index = False, header=None)

            print(">>> Database extraction successful")
            extraction.log(processing_time,
                            province_name,
                            district_name,
                            facility_name,
                            latest_site_id,
                            facility,
                            first_timestamp,
                            last_timestamp,
                            db_size,
                            version,
                            "Database extraction successful",
                            ehr_type= ehr_type
                            )
            # os.remove(facility)
            end_time = time.time() - start_time
            
            print(">>> Data extraction took ",round(time.time()-db_start_time,2),"s")
            print("\n")
            print("...............................................................................................")
        except:
            size = extraction.getDbSize(facility)
            print(">>> Extraction error here")
            extraction.log(
                            filename= facility,
                            processing_time= "",
                            db_size= size,
                            comment="Extraction error",
                            ehr_type= ehr_type
                            )
            # extraction.moveFailedDB(facility)
            print("\n")
            print("...............................................................................................")


    # Deduplication..................................................................................................
    print(">>> deduplication")
    merged = pd.read_csv("demos.csv")
    merged.drop_duplicates(subset=['person_id', 'firstname'], keep = 'last',inplace=True)
    clusters = extraction.deduplication(merged)


    # Generate CBS ID ................................................................................................
    cluster_id_dictionary = {}
    seq_number = 350000
    def cbs_unique_id(record,seq_number):
        if record["person_id"] == "":
            cbs_id = "No Demographics"
        elif record["cluster id"] in cluster_id_dictionary.keys():
            cbs_id = cluster_id_dictionary.get(record["cluster id"])
        else:
            first_letter_first_name = str(record["firstname"])[0]
            last_letter_first_name = str(record["firstname"])[-1]
            first_letter_last_name = str(record["Lastname"])[0]
            last_letter_last_name = str(record["Lastname"])[-1]
            first_letter_of_sex = str(record["sex"])[0]
            birthdate = str(record["birthdate"]).replace("/","")
            sequential_number = str(seq_number).rjust(6,"0")
            cbs_id = first_letter_first_name + last_letter_first_name + "-" + first_letter_last_name + last_letter_last_name + "-" + first_letter_of_sex + "-" + birthdate + "-" + sequential_number
            cluster_id_dictionary[ record["cluster id"]] = cbs_id
        return cbs_id

    cbs_ids =[]
    for i,row in clusters.iterrows():
        if row["cluster id"] in cluster_id_dictionary.keys() or row["firstname"] == "No Name":
            seq_number = seq_number
        else:
            seq_number = seq_number + 1
        cbs_ids.append(cbs_unique_id(row,seq_number))
    clusters["CBS ID"] = cbs_ids

    clusters.sort_values(by = ["cluster id"],inplace = True)

    clusters = clusters [[
        "person_id","CBS ID"
    ]]


    # Merging Facility Files////////////////////////////////////////////////////////////////////////////////////////////////////

    print(">>> Merging Files ")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_sex")]
    df_people_registered_in_ehr_sex =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_sex = df_people_registered_in_ehr_sex.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_sex.to_csv("people_registered_in_ehr_sex.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_education")]
    df_people_registered_in_ehr_education =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_education = df_people_registered_in_ehr_education.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_education.to_csv("people_registered_in_ehr_education.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_marital")]
    df_people_registered_in_ehr_marital =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_marital = df_people_registered_in_ehr_marital.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_marital.to_csv("people_registered_in_ehr_marital.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_nationality")]
    df_people_registered_in_ehr_nationality =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_nationality = df_people_registered_in_ehr_nationality.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_nationality.to_csv("people_registered_in_ehr_nationality.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_age")]
    df_people_registered_in_ehr_age =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_age = df_people_registered_in_ehr_age.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_age.to_csv("people_registered_in_ehr_age.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_occupation")]
    df_people_registered_in_ehr_occupation =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_occupation = df_people_registered_in_ehr_occupation.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_occupation.to_csv("people_registered_in_ehr_occupation.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_people_registered_in_ehr_religion")]
    df_people_registered_in_ehr_religion =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_people_registered_in_ehr_religion = df_people_registered_in_ehr_religion.append(df_temp,ignore_index = True)
    df_people_registered_in_ehr_religion.to_csv("people_registered_in_ehr_religion.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_diagnosis")]
    df_diagnosis =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_diagnosis = df_diagnosis.append(df_temp,ignore_index = True)
    df_diagnosis.to_csv("diagnosis_by_year.csv")


    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_diagnosis_by_year")]
    df_diagnosis_by_year =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_diagnosis_by_year = df_diagnosis_by_year.append(df_temp,ignore_index = True)
    df_diagnosis_by_year.to_csv("diagnosis_by_year.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_patients_attended_by_year")]
    df_patients_attended_by_year =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_patients_attended_by_year = df_patients_attended_by_year.append(df_temp,ignore_index = True)
    df_patients_attended_by_year.to_csv("patients_attended_by_year.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_distinct_patients_attended_by_year")]
    df_distinct_patients_attended_by_year =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_distinct_patients_attended_by_year = df_distinct_patients_attended_by_year.append(df_temp,ignore_index = True)
    df_distinct_patients_attended_by_year.to_csv("distinct_patients_attended_by_year.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_distinct_people_who_had_hts")]
    df_distinct_people_who_had_hts =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_distinct_people_who_had_hts = df_distinct_people_who_had_hts.append(df_temp,ignore_index = True)
    df_distinct_people_who_had_hts.to_csv("distinct_people_who_had_hts.csv")

    csv_files = glob.glob('./Aggregates/*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("c_distinct_people_who_had_hts_positive")]
    df_distinct_people_who_had_hts_positive =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_distinct_people_who_had_hts_positive = df_distinct_people_who_had_hts_positive.append(df_temp,ignore_index = True)
    df_distinct_people_who_had_hts_positive.to_csv("distinct_people_who_had_hts_positive.csv")


    csv_files = glob.glob('*.{}'.format('csv'))
    csv_files = [file for file in csv_files if file.startswith("CBS")]
    df_append =pd.DataFrame()
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_append = df_append.append(df_temp,ignore_index = True)
    cbs_id_dict = dict(zip(clusters.person_id, clusters["CBS ID"]))
    df_append['CBS ID'] = df_append['person_id'].map(cbs_id_dict)
    # shift column 'CBS ID' to first position
    first_column = df_append.pop('CBS ID')
    # insert column using insert(position,column_name,first_column) function
    df_append.insert(0, 'CBS ID', first_column)
    # Drop columns with identifiers
    df_append.loc[:,~df_append.columns.str.startswith('person_id')]
    
    df_append.drop_duplicates(keep= 'last', inplace=True)
    df_append = df_append.dropna(axis=0, subset=['Event date'])
    df_append.sort_values(by = ["Facility","CBS ID","Event date"],inplace = True)


    filename = datetime.today().strftime('%d%m%Y')
    #df_append = df_append.drop('person_id', axis=1)
    df_append.to_csv("CBS_" + filename + ".csv",index = False)
    
    df_append.to_excel("CBS_" + filename + ".xlsx")
    print(">>> Program completed")
    